<a href="https://colab.research.google.com/github/toche7/AI_ITM/blob/main/DemoVLLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab Demo vLLM server

In [ ]:
!nvidia-smi

เซลล์นี้รันคำสั่ง `nvidia-smi` เพื่อแสดงข้อมูลเกี่ยวกับ GPU ของ NVIDIA ที่มีอยู่ในสภาพแวดล้อม ซึ่งรวมถึงเวอร์ชันไดรเวอร์, เวอร์ชัน CUDA, การใช้งาน GPU และการใช้งานหน่วยความจำ สิ่งนี้มีประโยชน์สำหรับการตรวจสอบว่ามี GPU พร้อมใช้งานและทำงานได้อย่างถูกต้อง

In [ ]:
!pip install vllm pyngrok

เซลล์นี้ติดตั้งไลบรารี Python ที่จำเป็น: `vllm` สำหรับการให้บริการ LLM อย่างมีประสิทธิภาพ และ `pyngrok` สำหรับการสร้างอุโมงค์ที่ปลอดภัยเพื่อเปิดเผยบริการภายในเครื่องสู่สาธารณะบนอินเทอร์เน็ต

In [ ]:
!pip uninstall -y torchaudio

เซลล์นี้ถอนการติดตั้ง `torchaudio` ขั้นตอนนี้มักจะรวมอยู่ในการตั้งค่าที่เกี่ยวข้องกับ `vllm` เพื่อป้องกันความขัดแย้งที่อาจเกิดขึ้นหรือปัญหาเวอร์ชันกับไลบรารี PyTorch อื่นๆ ที่ `vllm` อาจต้องการ

In [ ]:
!nohup vllm serve Qwen/Qwen3-8B \
  --host 0.0.0.0 \
  --port 8000 \
  --api-key demo-key \
  > /content/vllm.log 2>&1 &

เซลล์นี้เริ่มต้นเซิร์ฟเวอร์ `vllm` ในพื้นหลังโดยใช้ `nohup` มันให้บริการโมเดล `Qwen/Qwen3-8B` โดยผูกกับโฮสต์ `0.0.0.0` และพอร์ต `8000` มีการตั้งค่า API key เป็น `demo-key` และเอาต์พุตทั้งหมดจะถูกเปลี่ยนเส้นทางไปยัง `/content/vllm.log`

In [ ]:
from google.colab import userdata

ngrok_token = userdata.get("NGROK_AUTH_TOKEN")
print(ngrok_token is not None)

เซลล์นี้ดึง `NGROK_AUTH_TOKEN` จากข้อมูลลับของผู้ใช้ Colab จากนั้นจะพิมพ์ `True` หากโทเค็นถูกดึงมาสำเร็จ (นั่นคือไม่ใช่ `None`) ซึ่งบ่งชี้ว่า ngrok ได้รับการกำหนดค่าด้วยโทเค็นการยืนยันตัวตนแล้ว

In [ ]:
from google.colab import userdata
from pyngrok import ngrok

ngrok.set_auth_token(userdata.get("NGROK_AUTH_TOKEN"))

public_url = ngrok.connect(8000)
print(public_url)

เซลล์นี้ตั้งค่าโทเค็นการยืนยันตัวตนของ ngrok จากนั้นสร้างอุโมงค์สาธารณะไปยังพอร์ต `8000` (ซึ่งเป็นที่ที่เซิร์ฟเวอร์ `vllm` กำลังทำงานอยู่) ตัวแปร `public_url` เก็บออบเจกต์อุโมงค์ ngrok และสตริง URL สาธารณะของมันจะถูกพิมพ์ ซึ่งสามารถใช้เพื่อเข้าถึงบริการ `vllm` จากภายนอกสภาพแวดล้อม Colab

In [ ]:
def create_curl_command(public_url, message):
    """
    Generates a curl command string to test the vllm server via ngrokTunnel.

    Args:
        public_url (str): The public URL from ngrokTunnel.
        message (str): The content of the user's message for the chat completion.

    Returns:
        str: The formatted curl command string.
    """
    # Ensure the public_url is a string (if it's an NgrokTunnel object, get its public_url attribute)
    if hasattr(public_url, 'public_url'):
        ngrok_url = public_url.public_url
    else:
        ngrok_url = public_url

    curl_command = f"""curl {ngrok_url}/v1/chat/completions \\
  -H \"Authorization: Bearer demo-key\" \\
  -H \"Content-Type: application/json\" \\
  -d '{{
    \"model\": \"Qwen/Qwen3-8B\",
    \"messages\": [
      {{\"role\": \"user\", \"content\": \"{message}\"}}
    ]
  }}'"""
    return curl_command

เซลล์นี้กำหนดฟังก์ชัน Python ชื่อ `create_curl_command` ฟังก์ชันนี้รับ URL สาธารณะของ ngrok และข้อความเป็นอินพุต และสร้างสตริงคำสั่ง `curl` คำสั่ง `curl` ได้รับการออกแบบมาเพื่อโต้ตอบกับปลายทาง API การเติมข้อความแชทของเซิร์ฟเวอร์ vLLM ทำให้ผู้ใช้สามารถทดสอบโมเดลที่ปรับใช้แล้วจากเทอร์มินัลหรือพรอมต์คำสั่ง

In [ ]:
# Demonstrate the function using the previously obtained public_url
# Assuming 'public_url' is available from cell ZNSKQTZeP6kJ

user_message = "Hello from my laptop, using a generated curl!"
generated_curl = create_curl_command(public_url, user_message)

print(generated_curl)

เซลล์นี้สาธิตการใช้งานฟังก์ชัน `create_curl_command` มันตั้งค่าข้อความตัวอย่างของผู้ใช้ เรียกใช้ฟังก์ชันด้วย `public_url` ที่ได้รับจาก ngrok จากนั้นพิมพ์สตริงคำสั่ง `curl` ที่เป็นผลลัพธ์ไปยังคอนโซล

curl abc123.ngrok-free.app/v1/chat/completions \
  -H "Authorization: Bearer demo-key" \
  -H "Content-Type: application/json" \
  -d '{
    "model": "Qwen/Qwen3-8B",
    "messages": [
      {"role": "user", "content": "Hello from my laptop"}
    ]
  }'

เซลล์นี้ส่งคำขอ POST ไปยังบริการ `vllm` ผ่าน `ngrokTunnel` โดยใช้ไลบรารี `requests` มันสร้าง URL API ตั้งค่าส่วนหัวที่จำเป็น (รวมถึงการอนุญาตด้วย `demo-key`) และส่งเพย์โหลด JSON พร้อมข้อความผู้ใช้ จากนั้นจะพิมพ์รหัสสถานะ HTTP และการตอบสนอง JSON จากเซิร์ฟเวอร์ `vllm` ทำให้สามารถทดสอบโมเดลที่ปรับใช้แล้วได้โดยทางโปรแกรม